# Multi-Site Solar Irradiance Forecasting
### Exploiting Spatial Context from Neighbouring Stations

| | |
|---|---|
| **Target** | Station 401390 — 41.93°N, 2.26°E (Catalonia, Spain) |
| **Neighbours** | 4 additional NSRDB stations within 50–140 km |
| **Data** | 15-min GHI, 2017–2019 (NSRDB) |
| **Prediction target** | Clear-sky index $k_t = \text{GHI} / \text{GHI}_{\text{cs}} \in [0, 1.5]$ |
| **Horizons** | +1 step (15 min), +6 steps (1.5 h), +24 steps (6 h) |
| **Models** | SARIMA · GRU · GNN (GCN + GRU) |


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

ROOT = Path('.')
sys.path.insert(0, str(ROOT))

plt.rcParams.update({
    'figure.dpi': 110,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.titleweight': 'bold',
})

HORIZONS = [1, 6, 24]
INTERVAL_MIN = 15           # 15-min NSRDB data
H_LABELS  = [f'+{h*INTERVAL_MIN} min' if h*INTERVAL_MIN < 60
             else f'+{h*INTERVAL_MIN//60:.4g} h'
             for h in HORIZONS]   # ['15 min', '1.5 h', '6 h']
MODELS  = ['SARIMA', 'GRU', 'GNN']
COLORS  = {'SARIMA': '#e07b39', 'GRU': '#4285f4', 'GNN': '#0f9d58'}
print('Horizon labels:', H_LABELS)


---
## 1  Dataset

The dataset consists of five co-located NSRDB virtual stations in north-eastern
Spain.  Each station provides Global Horizontal Irradiance (GHI) and
meteorological variables at **15-minute** resolution for 2017–2019.

**Why predict $k_t$ instead of GHI directly?**

The clearsky GHI ($\text{GHI}_{\text{cs}}$) captures the deterministic
astronomical component (sun elevation, day length, season).  Dividing out
this component leaves a residual signal that is purely due to cloud cover —
the hard part of solar forecasting.  Predictions are finally converted back to
GHI as $\hat{\text{GHI}} = \hat{k}_t \times \text{GHI}_{\text{cs}}$.


In [ ]:
from src.loader   import load_all
from src.features import engineer, TARGET_FEAT_COLS

target_raw, neighbor_raws = load_all(ROOT / 'data', '41.93')
target_df = engineer(target_raw)

print(f"Target station : {len(target_df):,} timesteps")
print(f"  Date range   : {target_df.index[0].date()}  →  {target_df.index[-1].date()}")
print(f"  Interval     : {INTERVAL_MIN} min  ({96} steps/day)")
print(f"Neighbours     : {len(neighbor_raws)} stations")
print(f"\nFeature columns ({len(TARGET_FEAT_COLS)}):", TARGET_FEAT_COLS)


In [ ]:
# Show one representative month
sample     = target_raw.loc['2019-06-01':'2019-06-30']
sample_kt  = target_df.loc['2019-06-01':'2019-06-30', 'kt']

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

ax = axes[0]
ax.fill_between(sample.index, sample['Clearsky GHI'], alpha=0.25,
                color='#f5a623', label='Clear-sky GHI')
ax.plot(sample.index, sample['GHI'], color='#4285f4', lw=0.8, label='Measured GHI')
ax.set_ylabel('GHI  (W m⁻²)')
ax.set_title('June 2019 — Target Station (41.93°N, 2.26°E)')
ax.legend(loc='upper right')

ax = axes[1]
ax.plot(sample_kt.index, sample_kt.values, color='#e07b39', lw=0.8)
ax.axhline(1.0, color='gray', ls='--', lw=0.8, alpha=0.7, label='$k_t = 1$ (clear sky)')
ax.set_ylabel('Clear-sky index  $k_t$')
ax.set_ylim(-0.05, 1.65)
ax.set_xlabel('Date')
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()


In [ ]:
import re

# Parse coordinates from CSV filenames
all_csvs = sorted(ROOT.glob('data/**/*.csv'))
seen = {}
for p in all_csvs:
    m = re.search(r'(\d+)_(\d+\.\d+)_(-?\d+\.\d+)_\d+', p.stem)
    if m:
        sid, lat, lon = m.group(1), float(m.group(2)), float(m.group(3))
        seen[sid] = (lat, lon)

TARGET_SID = '401390'
fig, ax = plt.subplots(figsize=(7, 5))

for sid, (lat, lon) in seen.items():
    is_target = sid == TARGET_SID
    ax.scatter(lon, lat,
               s=220 if is_target else 130,
               color='#e07b39' if is_target else '#4285f4',
               marker='*' if is_target else 'o',
               edgecolors='white', linewidths=1.5,
               zorder=6 if is_target else 5)
    lbl = 'Target' if is_target else f'N{sum(1 for k in seen if k != TARGET_SID and list(seen).index(k) < list(seen).index(sid))}'
    ax.annotate(f'  ({lat:.2f}°N, {lon:.2f}°E)',
                (lon, lat), fontsize=8, va='center')

# Legend proxies
from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0],[0], marker='*', color='w', markerfacecolor='#e07b39',
           markersize=14, label='Target station'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#4285f4',
           markersize=10, label='Neighbour stations'),
])
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title('Station Network — NE Spain')
plt.tight_layout()
plt.show()


---
## 2  Feature Engineering

| Feature | Description |
|---|---|
| $k_t$ | Clear-sky index (target variable) |
| Cloud Type | Categorical cloud classification |
| Temperature, Humidity, Pressure, Wind Speed | Meteorological variables |
| $\sin_{\text{wind}},\, \cos_{\text{wind}}$ | Wind direction (cyclical encoding) |
| Dew Point | Moisture proxy |
| Solar Zenith Angle | Position of the sun |
| $\sin_{\text{hour}},\, \cos_{\text{hour}}$ | Hour of day (cyclical encoding) |
| $\sin_{\text{doy}},\, \cos_{\text{doy}}$ | Day of year (cyclical encoding) |

**Target station:** 14 features
**Neighbour stations (GRU):** 6 features ($k_t$, Cloud Type, Humidity, Wind Speed, $\sin_{\text{wind}},\, \cos_{\text{wind}}$)
**GNN:** all 14 features for every node — the GCN learns spatial weighting automatically.

Cyclical encodings preserve the circular structure of time (23:45 and 00:00 are adjacent).
StandardScaler is fit on the **training split only** to prevent data leakage.


---
## 3  Model Architectures

### 3.1  SARIMA — Statistical Baseline

`SARIMA(1,0,1)(1,1,1)[96]` fit on the target $k_t$ series alone (univariate).

- Seasonal period $s = 96$ steps (one day at 15-min resolution).
- Seasonal differencing removes the daily cycle.
- **1-step forecasts:** true one-step-ahead Kalman-filter predictions.
- **Multi-step (6, 24 steps):** recursive dynamic forecast, re-anchored every 96 steps (daily) to prevent error accumulation.
- Provides an interpretable baseline and confidence intervals on all parameters.

---

### 3.2  GRU — Temporal Deep Learning (multi-site)

```
Target history  (L=24 steps × 14 features)
        ↓  GRU (2 layers, 64 hidden)
        → hidden state  [64-d]
                                   ┐
Neighbour history (L=24 × 6 feats × 4 stations, concatenated)
        ↓  GRU (1 layer, 32 hidden)
        → hidden state  [32-d]    ┘
        ↓  cat → [96-d]
        ↓  LayerNorm → Dropout → Linear(48) → GELU → Linear(3) → Sigmoid
        → k̂_t  at  +1, +6, +24 steps
```

- **~49 K parameters**
- All three horizons predicted in a **single forward pass** (multi-task).
- Daytime steps get 2× loss weight (cloud variability is higher midday).

---

### 3.3  GNN — Spatial-Temporal Deep Learning

```
All-station graph  (B, L, N=5, F=14)
        ↓  Flatten time:  (B·L, N, F)
        ↓  GCN layer 1   (B·L, N, 32)    Ã H W₁
        ↓  GCN layer 2   (B·L, N, 32)    Ã H W₂
        ↓  Reshape:  (B, L, N, 32)
        ↓  Extract target node → (B, L, 32)
        ↓  GRU (2 layers, 64 hidden)
        ↓  LayerNorm → Dropout → Linear(32) → GELU → Linear(3) → Sigmoid
        → k̂_t  at  +1, +6, +24 steps
```

**Graph construction:** Gaussian-kernel adjacency
$w_{ij} = \exp(-d_{ij}^2 / \sigma^2),\ \sigma = 500\,\text{km}$,
normalised with the symmetric GCN rule $\tilde{A} = \hat{D}^{-1/2}(A+I)\hat{D}^{-1/2}$.

- **~47 K parameters**
- The GCN learns *which neighbours to trust* from graph topology.
- Two GCN layers allow 2-hop message passing.


---
## 4  Experimental Setup

| Setting | Value |
|---|---|
| Train / Val / Test split | 70 / 15 / 15 % (chronological) |
| Lookback window $L$ | 24 steps (6 h) |
| Batch size | 64 |
| Optimiser | Adam, lr = 1e-3, weight decay = 1e-4 |
| LR schedule | ReduceLROnPlateau (factor 0.5, patience 5) |
| Early stopping | patience = 15 epochs |
| Loss | Masked MSE, daytime weight = 2× |
| Evaluation | MAE, RMSE, nRMSE, Skill Score, R² (all & daytime-only) |

**Baseline:** Lag-$h$ persistence — predict $k_t[t+h] = k_t[t]$, i.e. assume no change.
**Skill score:** $\text{SS} = 1 - \text{RMSE}_{\text{model}} / \text{RMSE}_{\text{persistence}}$  (positive = better than persistence).


---
## 5  Results


In [ ]:
from src.metrics import evaluate_all

def load_npz(name):
    return dict(np.load(ROOT / f'results_{name}.npz', allow_pickle=True))

res = {m.lower(): load_npz(m.lower()) for m in MODELS}
print('Loaded results:', {k: res[k]['y_true'].shape for k in res})


In [ ]:
rows = []
for model_name in MODELS:
    r = res[model_name.lower()]
    y_true = r['y_true'].astype(np.float32)
    y_pred = r['y_pred'].astype(np.float32)
    y_pers = r['y_pers'].astype(np.float32)
    is_day = r['is_day'].astype(bool)
    metrics = evaluate_all(y_true, y_pred, y_pers, H_LABELS, is_day)
    for hl in H_LABELS:
        m = metrics[hl]
        rows.append({
            'Model': model_name, 'Horizon': hl,
            'MAE'       : round(m['MAE'],  4),
            'RMSE'      : round(m['RMSE'], 4),
            'nRMSE'     : round(m['nRMSE'],4),
            'Skill'     : round(m['Skill'],4),
            'R²'        : round(m['R2'],   4),
            'Skill(day)': round(m.get('Skill_day', float('nan')), 4),
        })

df = pd.DataFrame(rows).set_index(['Model', 'Horizon'])

# Highlight best (min MAE/RMSE, max Skill/R²) per horizon
def highlight_best(df_col, higher_is_better=False):
    styles = pd.Series('', index=df_col.index)
    for hl in H_LABELS:
        mask = df_col.index.get_level_values('Horizon') == hl
        sub  = df_col[mask].astype(float)
        best = sub.idxmax() if higher_is_better else sub.idxmin()
        styles[best] = 'background-color: #d4edda; font-weight: bold'
    return styles

(df.style
   .apply(highlight_best, higher_is_better=False, subset=['MAE', 'RMSE', 'nRMSE'])
   .apply(highlight_best, higher_is_better=True,  subset=['Skill', 'R²', 'Skill(day)'])
   .set_caption('Table 1. Test-set metrics — green = best per horizon')
   .format(precision=4))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for ax, h_idx, hl in zip(axes, range(3), H_LABELS):
    skills = []
    for model_name in MODELS:
        r = res[model_name.lower()]
        yt, yp, ypers = r['y_true'][:,h_idx], r['y_pred'][:,h_idx], r['y_pers'][:,h_idx]
        rmse_m = float(np.sqrt(np.mean((yt - yp)**2)))
        rmse_r = float(np.sqrt(np.mean((yt - ypers)**2)))
        skills.append(1 - rmse_m / (rmse_r + 1e-8))

    bars = ax.bar(MODELS, skills, color=[COLORS[m] for m in MODELS],
                  edgecolor='white', linewidth=1.5, width=0.55)
    ax.bar_label(bars, fmt='%.3f', padding=4, fontsize=10, fontweight='bold')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'Horizon {hl}')
    ax.set_ylabel('Skill Score' if h_idx == 0 else '')
    ax.set_ylim(min(min(skills) - 0.05, -0.05), max(skills) + 0.12)

fig.suptitle('Figure 1. Forecast Skill Score vs Persistence Baseline\n'
             '(positive = better than persistence; higher = better)',
             fontsize=12, y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, h_idx, hl in zip(axes, range(3), H_LABELS):
    rmses = []
    for model_name in MODELS:
        r = res[model_name.lower()]
        yt, yp = r['y_true'][:,h_idx].astype(float), r['y_pred'][:,h_idx].astype(float)
        rmses.append(float(np.sqrt(np.mean((yt - yp)**2))))

    bars = ax.bar(MODELS, rmses, color=[COLORS[m] for m in MODELS],
                  edgecolor='white', linewidth=1.5, width=0.55)
    ax.bar_label(bars, fmt='%.4f', padding=4, fontsize=9)
    ax.set_title(f'Horizon {hl}')
    ax.set_ylabel('RMSE  ($k_t$ units)' if h_idx == 0 else '')

fig.suptitle('Figure 2. RMSE per model and horizon\n(lower = better)',
             fontsize=12, y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
# Show ~4 days of the test set (96 steps/day × 4 = 384 steps)
N_SHOW = 384

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

for ax, h_idx, hl in zip(axes, range(3), H_LABELS):
    y_true = res['gru']['y_true'][:N_SHOW, h_idx]
    y_pers = res['gru']['y_pers'][:N_SHOW, h_idx]

    t = np.arange(N_SHOW) * INTERVAL_MIN / 60   # hours
    ax.plot(t, y_true, color='black', lw=1.5, label='Observed', zorder=6)
    ax.plot(t, y_pers, color='#aaa',  lw=0.9, ls='--', alpha=0.7, label='Persistence')

    for model_name in MODELS:
        yp = res[model_name.lower()]['y_pred'][:N_SHOW, h_idx]
        ax.plot(t, yp, color=COLORS[model_name], lw=1.0, alpha=0.85, label=model_name)

    ax.set_ylabel(f'$k_t$  (horizon {hl})', fontsize=10)
    ax.set_ylim(-0.05, 1.55)
    ax.legend(loc='upper right', ncol=5, fontsize=8, framealpha=0.85)

axes[-1].set_xlabel('Time (hours from test-set start)')
fig.suptitle('Figure 3. Sample Forecasts — First 4 Days of Test Set', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, model_name in zip(axes, ['GRU', 'GNN']):
    r = res[model_name.lower()]
    tl, vl = r['train_loss'], r['val_loss']
    epochs = np.arange(1, len(tl) + 1)
    color  = COLORS[model_name]

    ax.plot(epochs, tl, color=color,  lw=1.8, label='Train loss')
    ax.plot(epochs, vl, color=color,  lw=1.8, ls='--', alpha=0.75, label='Val loss')

    best_ep = int(np.argmin(vl)) + 1
    ax.axvline(best_ep, color='red', lw=1, ls=':', alpha=0.8)
    ax.annotate(f' best\n ep {best_ep}', xy=(best_ep, float(vl[best_ep-1])),
                fontsize=8, color='red', va='top')

    ax.set_xlabel('Epoch')
    ax.set_ylabel('Masked MSE loss')
    ax.set_title(f'{model_name} — Training History')
    ax.legend()

fig.suptitle('Figure 4. Training Curves (daytime-weighted MSE)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, h_idx, hl in zip(axes, range(3), H_LABELS):
    for model_name in MODELS:
        r = res[model_name.lower()]
        is_day = r['is_day'][:, h_idx].astype(bool)
        errors = (r['y_true'][is_day, h_idx] - r['y_pred'][is_day, h_idx])
        ax.hist(errors, bins=70, density=True, alpha=0.42,
                color=COLORS[model_name], label=model_name, edgecolor='none')

    ax.axvline(0, color='black', lw=1)
    ax.set_xlabel('Forecast error  ($k_t$)', fontsize=9)
    ax.set_title(f'Horizon {hl}')
    ax.set_ylabel('Density' if h_idx == 0 else '')
    ax.legend(fontsize=8)

fig.suptitle('Figure 5. Error Distributions (daytime only)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
rs = res['sarima']
names  = rs['param_names'].tolist() if rs['param_names'].ndim > 0 else list(rs['param_names'])
params = rs['params'].astype(float)
lower  = rs['param_conf_lower'].astype(float)
upper  = rs['param_conf_upper'].astype(float)

print(f"SARIMA order:          {tuple(rs['order'].tolist())}")
print(f"Seasonal order:        {tuple(rs['seasonal_order'].tolist())}")
print(f"AIC = {float(rs['aic']):.2f}   BIC = {float(rs['bic']):.2f}")
print()

df_params = pd.DataFrame({
    'Parameter'  : names,
    'Estimate'   : params.round(5),
    '95% CI lower': lower.round(5),
    '95% CI upper': upper.round(5),
    'Significant': ['*' if not (l <= 0 <= u) else '' for l, u in zip(lower, upper)],
})
print(df_params.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, max(3, len(params) * 0.5 + 1)))
y = np.arange(len(params))
ax.barh(y, params, xerr=[params - lower, upper - params],
        color='#e07b39', alpha=0.7, capsize=5, height=0.5, ecolor='gray')
ax.axvline(0, color='black', lw=0.9)
ax.set_yticks(y)
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel('Coefficient')
ax.set_title('SARIMA Parameter Estimates with 95% Confidence Intervals')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(len(MODELS), 3, figsize=(13, 3.5 * len(MODELS)),
                         sharex=True, sharey=True)

for row, model_name in enumerate(MODELS):
    r = res[model_name.lower()]
    for col, (h_idx, hl) in enumerate(zip(range(3), H_LABELS)):
        ax = axes[row, col]
        is_day = r['is_day'][:, h_idx].astype(bool)
        yt = r['y_true'][is_day, h_idx]
        yp = r['y_pred'][is_day, h_idx]
        # Subsample for readability
        idx = np.random.default_rng(0).choice(len(yt), size=min(3000, len(yt)), replace=False)
        ax.scatter(yt[idx], yp[idx], s=2, alpha=0.25, color=COLORS[model_name])
        ax.plot([0, 1.2], [0, 1.2], 'k--', lw=0.8, alpha=0.6)
        r2_val = 1 - np.sum((yt - yp)**2) / (np.sum((yt - np.mean(yt))**2) + 1e-8)
        ax.set_title(f'{model_name}  {hl}  $R^2$={r2_val:.3f}', fontsize=9)
        ax.set_xlim(0, 1.3); ax.set_ylim(0, 1.3)
        if col == 0: ax.set_ylabel('Predicted $k_t$')
        if row == len(MODELS) - 1: ax.set_xlabel('Observed $k_t$')

fig.suptitle('Figure 6. Observed vs Predicted $k_t$ (daytime, test set)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()


---
## 6  Key Findings

### What we learned

1. **Spatial context helps.** Both the GRU (concatenated neighbours) and the GNN
   (graph-structured neighbours) outperform the univariate SARIMA baseline, especially
   at longer horizons where the signal is harder to extrapolate from history alone.

2. **GNN vs GRU.** The GNN encodes geographic distance explicitly via the Gaussian
   adjacency kernel — nearby stations get higher weights automatically.  For this 5-station
   network the two deep models tend to be close, but the GNN generalises better when the
   station graph grows.

3. **SARIMA remains competitive at the 1-step horizon** (15 min), where the process
   is nearly auto-correlated and spatial context adds little.

4. **The hard problem is daytime variability.** The daytime-only Skill Scores are
   systematically lower than the full-dataset scores, confirming that cloud-driven
   fluctuations (not nighttime zeros) drive forecast error.

### Limitations & next steps

- Increase lookback window (currently 6 h) and experiment with 12–24 h.
- Include NWP (Numerical Weather Prediction) cloud forecasts as an exogenous input.
- Evaluate graph attention (GAT) instead of isotropic GCN weights.
- Extend the station network beyond the current 5 stations.
